# GPR151 Drug Discovery Pipeline — Kaggle Edition

N-Terminal Truncation -> Membrane MD -> GUM Docking -> Key Residues

**Target:** GPR151 (UniProt Q8TDV0) — orphan GPCR, habenula/DRG/spinal cord
**Strategy:** Remove autoinhibitory N-terminus -> relax orthosteric pocket via MD -> dock GUM-class psoralen agonists
**Pipeline:** AlphaFold structure -> GPCRDB TM annotation -> truncation -> membrane embedding -> OpenMM MD -> pocket analysis -> AutoDock Vina

## Why this is a Kaggle notebook and not the original Colab one

Ported after the Colab free-tier run hit three structural problems:

1. **condacolab's conda env didn't match the kernel's Python** (packages landed in a python3.11 site-packages dir while the live kernel ran python3.12 — every `import openmm`/`import parmed` after the conda step failed or hung for 15+ minutes).
2. **The runtime disconnected/crashed repeatedly**, wiping `/content` (no persistent disk) and killing a 50 ns MD run that needs far longer than one sitting.
3. **`Run all` does not survive a browser tab closing** on the free tier — the run was only as durable as the babysitting.

Kaggle fixes (1) and (3) structurally: a single pip-installable Python env (nothing to mismatch), and **Save Version -> Save & Run All (Commit)** executes server-side even with the browser closed. It does **not** remove (2)'s underlying constraint — Kaggle GPU sessions cap at ~9h and the weekly GPU quota is ~30h — so SECTION 8 (production MD) is checkpointed across sessions rather than assuming one continuous run, the same way the original notebook checkpointed to Google Drive.

**Runtime requirement:** Settings (right sidebar) -> Accelerator -> GPU T4 x2 or GPU P100, and Settings -> Internet -> On. Both together require a phone-verified Kaggle account.
**Estimated time:** same compute as Colab (~3-4h for setup through equilibration; the 50 ns production run is realistically several ~9h sessions — see SECTION 8).

---
**Key references:**
- Wang et al. 2026 PNAS — First GPR151 structure (NELiS platform), autoinhibitory N-terminus confirmed. DOI: 10.1073/pnas.2534234123
- Yoshida et al. 2026 Genes to Cells — GUM3/GUM4 psoralen agonists. DOI: 10.1111/gtc.70119


## SECTION 0: Environment Setup

Two install paths, deliberately kept apart — this is the actual fix for the Colab crash:

- **pip**, straight into the kernel's own Python — OpenMM, ParmEd, MDAnalysis, BioPython, RDKit, Vina, ProLIF, etc. Kaggle has one Python env, so there's nothing for these to mismatch against.
- **Miniconda in userspace**, used only to get the `packmol-memgen`/`tleap` CLI binaries from the `ambermd` conda channel (AmberTools has no pip wheel). We never `import` from this env in the kernel — only call its binaries as subprocesses — so there's no mismatch possible here either.

No restart needed for either step.


In [ ]:
import os, shutil

# Find the actual dataset path
src = '/kaggle/input/datasets/eric847/gpr151-checkpoint'
if not os.path.exists(src):
    # Sometimes Kaggle appends a version number
    for item in os.listdir('/kaggle/input/'):
        if 'gpr151' in item.lower():
            src = os.path.join('/kaggle/input/', item)
            print(f'Found dataset at: {src}')
            break

if not os.path.exists(src):
    raise FileNotFoundError('Cannot find gpr151-checkpoint dataset')

# Copy everything to working directory
for root, dirs, files in os.walk(src):
    for f in files:
        s = os.path.join(root, f)
        rel = os.path.relpath(s, src)
        d = os.path.join('/kaggle/working', rel)
        os.makedirs(os.path.dirname(d), exist_ok=True)
        shutil.copy2(s, d)

print('✅ Files restored')
for item in sorted(os.listdir('/kaggle/working')):
    print(f'  {item}')

In [ ]:
import os

# Check what's actually in /kaggle/input/
print("Contents of /kaggle/input/:")
for item in sorted(os.listdir('/kaggle/input/')):
    path = os.path.join('/kaggle/input/', item)
    if os.path.isdir(path):
        print(f"  📁 {item}/")
        # Show what's inside
        for sub in os.listdir(path)[:10]:
            print(f"      {sub}")
    else:
        print(f"  📄 {item}")

In [ ]:
import os
for f in ['gpr151_md.dcd', 'gpr151_md.checkpoint', 'gpr151_pipeline/system.prmtop']:
    p = os.path.join('/kaggle/working', f)
    if os.path.exists(p):
        print(f'✅ {f}: {os.path.getsize(p)/1024/1024:.1f} MB')
    else:
        print(f'❌ {f} missing')

In [ ]:
#2
# Step 0a: pip-install packages that have wheels, directly into the kernel's Python.
# NOTE: We do NOT pip install openmm here — Kaggle GPU containers pre-install
# openmm with CUDA support. pip install openmm would OVERWRITE it with a
# non-CUDA pip build, killing GPU acceleration. The pre-installed version is correct.
import sys, subprocess, time, os

t0 = time.time()
print('Installing for:', sys.executable)
# Pin openmm to Kaggle's pre-installed CUDA version BEFORE installing deps.
# pdbfixer and openmmforcefields depend on openmm — without pinning,
# pip may upgrade openmm to a non-CUDA build, breaking GPU acceleration.
try:
    import openmm as _omm_check
    _omm_pin = f'openmm=={_omm_check.__version__}'
    print(f'Pinning openmm at {_omm_check.__version__} to prevent upgrade')
except ImportError:
    _omm_pin = 'openmm'
    print('openmm not pre-installed, will install via pip')

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     _omm_pin, 'openmmforcefields', 'pdbfixer', 'parmed', 'py3Dmol',
     'mdanalysis', 'biopython', 'rdkit', 'vina', 'prolif',
     'matplotlib', 'seaborn', 'pandas', 'numpy', 'scipy'],
    check=True,
)
print(f'pip install done in {time.time()-t0:.0f}s')

# Verify OpenMM and try to get CUDA platform
# pip openmm 8.5.x wheels for python3.12 ship Reference+CPU only.
# Try installing the separate CUDA plugin package.
import openmm
from openmm import Platform
_platforms = [Platform.getPlatform(i).getName() for i in range(Platform.getNumPlatforms())]
print(f'OpenMM {openmm.__version__} platforms: {_platforms}')
if 'CUDA' not in _platforms and 'OpenCL' not in _platforms:
    print('Only CPU platforms found — attempting to add CUDA/OpenCL plugin...')
    _omm_ver = openmm.__version__
    # Try conda install of GPU-capable openmm into base (if miniconda exists)
    import os
    _conda = '/opt/miniconda/bin/conda'
    if os.path.exists(_conda):
        subprocess.run([_conda, 'install', '-y', '-q', '-n', 'base',
                        '-c', 'conda-forge', f'openmm={_omm_ver}', 'cuda-version=12'],
                       capture_output=True)
        # Add conda base to sys.path so we get the CUDA-enabled openmm
        import glob as _gl
        _conda_site = _gl.glob('/opt/miniconda/lib/python3*/site-packages')
        if _conda_site:
            import sys
            sys.path.insert(0, _conda_site[0])
            # Force reimport
            import importlib
            import openmm
            importlib.reload(openmm)
            from openmm import Platform
            _platforms = [Platform.getPlatform(i).getName() for i in range(Platform.getNumPlatforms())]
            print(f'After conda install — platforms: {_platforms}')
    if 'CUDA' not in _platforms:
        print('⚠️  CUDA not available. MD will run on CPU — very slow.')
        print('   Make sure GPU T4 x2 is selected in Kaggle Settings → Accelerator.')
if 'CUDA' in _platforms:
    print('✅ CUDA platform available — MD will use GPU acceleration')
elif 'OpenCL' in _platforms:
    print('⚠️  OpenCL available (no CUDA) — MD will be slower than optimal')

# Install Open Babel (needed for ligand/receptor PDBQT conversion in Section 10)
subprocess.run(['apt-get', 'install', '-y', '-q', 'openbabel'], check=True)
print('obabel installed')

In [ ]:
import importlib
packages = ['MDAnalysis', 'vina', 'ProLIF', 'rdkit', 'seaborn', 'numpy', 'matplotlib']
for pkg in packages:
    try:
        mod = importlib.import_module(pkg)
        print(f'✅ {pkg} already installed')
    except ImportError:
        print(f'❌ {pkg} needs installation')

In [ ]:
#3
# Step 0b: Miniconda in userspace, AmberTools only, for the packmol-memgen / tleap CLI.
# We deliberately do NOT activate this env or import from it in this kernel —
# only its binaries are called by full path via subprocess (SECTION 5 below).
#
# IMPORTANT: the conda prefix and installer MUST live OUTSIDE /kaggle/working.
# /kaggle/working is what Kaggle commits as the notebook's Output on Save & Run All —
# a full Miniconda + AmberTools env is thousands of files / 1-3+ GB, which blew the
# output size/count limit on Version 4 (cells ran fine, but the post-execution commit
# silently failed - logs showed nbconvert succeeding with no Python traceback).
import os, subprocess, time

CONDA_PREFIX = '/opt/miniconda'
AMBER_ENV = f'{CONDA_PREFIX}/envs/ambertools'

if not os.path.exists(f'{AMBER_ENV}/bin/packmol-memgen'):
    t0 = time.time()
    subprocess.run(['wget', '-q',
                     'https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh',
                     '-O', '/tmp/miniconda.sh'], check=True)
    subprocess.run(['bash', '/tmp/miniconda.sh', '-b', '-f', '-p', CONDA_PREFIX], check=True)
    os.remove('/tmp/miniconda.sh')
    conda_bin = f'{CONDA_PREFIX}/bin/conda'
    subprocess.run([conda_bin, 'tos', 'accept', '--override-channels', '--channel', 'https://repo.anaconda.com/pkgs/main'])
    subprocess.run([conda_bin, 'tos', 'accept', '--override-channels', '--channel', 'https://repo.anaconda.com/pkgs/r'])
    subprocess.run([conda_bin, 'create', '-y', '-q', '-n', 'ambertools',
                     '-c', 'conda-forge', '-c', 'ambermd', 'ambertools'], check=True)
    subprocess.run([conda_bin, 'clean', '-a', '-y'], check=True)  # drop package cache, keep output lean
    print(f'AmberTools env built in {time.time()-t0:.0f}s')
else:
    print('AmberTools env already present, skipping')

PACKMOL_MEMGEN = f'{AMBER_ENV}/bin/packmol-memgen'
assert os.path.exists(PACKMOL_MEMGEN), 'packmol-memgen not found — check the conda build log above'
print('packmol-memgen at:', PACKMOL_MEMGEN)


In [ ]:
#4
# Step 0c: Verify GPU and imports (should be instant — no conda/kernel mismatch to hang on)
import subprocess, shutil
if shutil.which('nvidia-smi'):
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                            capture_output=True, text=True)
    print('GPU:', result.stdout.strip())
else:
    print('⚠️  nvidia-smi not found — GPU accelerator may not be enabled.')
    print('   Go to Kaggle Settings (right sidebar) → Accelerator → GPU T4 x2, then re-run.')

import openmm
from openmm import Platform
for i in range(Platform.getNumPlatforms()):
    print(f'  OpenMM platform: {Platform.getPlatform(i).getName()}')

import os
os.makedirs('/kaggle/working/gpr151_pipeline', exist_ok=True)
print('\n✅ Environment ready')

## SECTION 0.5: Resume from a previous Kaggle session (optional)

Kaggle sessions cap at ~9h, so SECTION 8 (production MD) is checkpointed exactly like it was
checkpointed to Google Drive in the Colab version — same chunk size, same file names — just
carried between sessions via a Kaggle Dataset instead of a Drive folder:

1. After a commit finishes, open its **Output** tab -> **New Dataset** to publish
   `gpr151_pipeline/` (e.g. as `gpr151-checkpoint`).
2. In the next session, **Add Data -> Your Datasets -> gpr151-checkpoint** (mounts read-only
   at `/kaggle/input/gpr151-checkpoint/`).
3. Run the cell below to copy any checkpoint/trajectory files into the writable
   `/kaggle/working/gpr151_pipeline/` — SECTION 8 auto-detects `steps_completed.txt` and resumes.

Skip this cell on the first session.


In [ ]:
#6
import glob, shutil, os

RESUME_INPUT = '/kaggle/input/gpr151-checkpoint'  # rename to match your dataset slug
if os.path.isdir(RESUME_INPUT):
    copied = 0
    for f in glob.glob(f'{RESUME_INPUT}/**/*', recursive=True):
        if os.path.isfile(f):
            dest = f.replace(RESUME_INPUT, '/kaggle/working/gpr151_pipeline')
            os.makedirs(os.path.dirname(dest), exist_ok=True)
            shutil.copy2(f, dest)
            copied += 1
    print(f'✅ Restored {copied} files from previous session')
else:
    print('No resume dataset attached — starting fresh')


## SECTION 1: Fetch GPR151 AlphaFold Structure
We use the wild-type human GPR151 AlphaFold model (UniProt Q8TDV0) rather than the NELiS cryo-EM structure, which contains 4 thermostabilizing mutations and a bound nanobody. The WT AlphaFold model gives us the native sequence for drug discovery without confounding mutations.

In [ ]:
#8
import requests, os
os.chdir('/kaggle/working')
import os
import json

os.makedirs('gpr151_pipeline', exist_ok=True)

# Download AlphaFold v4 model for human GPR151 (UniProt Q8TDV0)
AF_URL = 'https://alphafold.ebi.ac.uk/files/AF-Q8TDV0-F1-model_v6.pdb'
print(f'Fetching AlphaFold structure: {AF_URL}')

r = requests.get(AF_URL, timeout=60)
r.raise_for_status()

with open('gpr151_pipeline/gpr151_alphafold.pdb', 'w') as f:
    f.write(r.text)

# Count residues
lines = [l for l in r.text.split('\n') if l.startswith('ATOM') and l[12:16].strip() == 'CA']
residue_count = len(lines)
print(f'✅ Downloaded GPR151 AlphaFold model: {residue_count} residues')

# Also fetch AlphaFold PAE (confidence) data to identify low-confidence N-terminal region
PAE_URL = 'https://alphafold.ebi.ac.uk/files/AF-Q8TDV0-F1-predicted_aligned_error_v6.json'
r_pae = requests.get(PAE_URL, timeout=60)
if r_pae.ok:
    pae_data = r_pae.json()
    print(f'✅ Downloaded PAE confidence data')
    with open('gpr151_pipeline/gpr151_pae.json', 'w') as f:
        json.dump(pae_data, f)
else:
    print('PAE data not available, will use pLDDT from PDB B-factors')

In [ ]:
#9
# Also fetch UniProt sequence and metadata for GPR151
UNIPROT_URL = 'https://rest.uniprot.org/uniprotkb/Q8TDV0.json'
r_up = requests.get(UNIPROT_URL, timeout=30)
r_up.raise_for_status()
uniprot_data = r_up.json()

sequence = uniprot_data['sequence']['value']
seq_length = uniprot_data['sequence']['length']
print(f'GPR151 sequence length: {seq_length} residues')
print(f'First 60 aa (N-terminus region): {sequence[:60]}')
print(f'Gene name: {uniprot_data["genes"][0]["geneName"]["value"]}')

# Extract annotated transmembrane regions from UniProt
tm_regions = []
for feature in uniprot_data.get('features', []):
    if feature.get('type') == 'Transmembrane':
        start = feature['location']['start']['value']
        end = feature['location']['end']['value']
        desc = feature.get('description', '')
        tm_regions.append((start, end, desc))
        print(f'  TM region: residues {start}–{end} ({desc})')

if tm_regions:
    TM1_START = tm_regions[0][0]  # First TM helix start
    print(f'\n📌 TM1 starts at residue {TM1_START}')
    print(f'   N-terminal autoinhibitory region: residues 1–{TM1_START - 1}')
else:
    print('UniProt TM annotation not found — will use GPCRDB in next cell')
    TM1_START = 42  # conservative fallback; overridden by GPCRDB or manual override below

## SECTION 2: GPCRDB TM Helix Annotation
Query the GPCRDB API for GPR151 generic numbering to precisely define TM1 start and validate the truncation boundary.

In [ ]:
#11
import requests
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Query GPCRDB for GPR151 residue annotations
GPCRDB_URL = 'https://gpcrdb.org/services/residues/extended/gpr151_human/'
print(f'Querying GPCRDB: {GPCRDB_URL}')

try:
    r_gpcr = requests.get(GPCRDB_URL, timeout=30)
    residues = r_gpcr.json()

    # Build dataframe of residues with segment annotation
    df_res = pd.DataFrame(residues)
    print(f'Retrieved {len(df_res)} residue annotations')
    if len(df_res) == 0:
        raise ValueError('GPCRDB returned 0 residues for gpr151_human')
    print(f'Segments found: {df_res["protein_segment"].unique().tolist()}')

    # Find TM1 boundaries
    tm1 = df_res[df_res['protein_segment'] == 'TM1']
    if len(tm1) > 0:
        GPCRDB_TM1_START = int(tm1['sequence_number'].min())
        GPCRDB_TM1_END = int(tm1['sequence_number'].max())
        print(f'\n📌 GPCRDB TM1: residues {GPCRDB_TM1_START}–{GPCRDB_TM1_END}')

        # Show all TM helices
        for tm in ['TM1','TM2','TM3','TM4','TM5','TM6','TM7']:
            seg = df_res[df_res['protein_segment'] == tm]
            if len(seg) > 0:
                print(f'  {tm}: {int(seg["sequence_number"].min())}–{int(seg["sequence_number"].max())}')

        # N-terminus (before TM1)
        nterm = df_res[df_res['protein_segment'] == 'N-term']
        print(f'\n  N-terminus annotated: {len(nterm)} residues (1–{int(nterm["sequence_number"].max()) if len(nterm)>0 else GPCRDB_TM1_START-1})')

        # Use GPCRDB annotation if available, else UniProt
        TM1_START = GPCRDB_TM1_START

    # Save for reference
    df_res.to_csv('gpr151_pipeline/gpcrdb_residues.csv', index=False)

except Exception as e:
    print(f'GPCRDB query failed: {e}')
    print('Falling back to UniProt TM annotation')
    # TM1_START may already be set from UniProt; if not, the manual override below handles it


# --- Manual override: cut at residue 32, not the sequence-topology-based TM1_START ---
# GPCRDB returned 0 residues for gpr151_human (query failed), and UniProt's
# Transmembrane(42-62) annotation is a sequence-based topology prediction
# (Phobius/TMHMM-style), not a measurement of the actual AlphaFold structure.
#
# A real per-residue check of the AlphaFold model (pLDDT + secondary structure,
# via biotite.structure.annotate_sse) shows:
#   - res 1-10:  pLDDT 35-43 (very low) -> genuinely disordered, likely the real
#                autoinhibitory arm that occludes the orthosteric pocket
#   - res 11-20: pLDDT 34-56, called "helix" by the SSE algorithm, but at this
#                confidence that call is not trustworthy (spurious)
#   - res 21-31: pLDDT 60-78, noisy transitional coil
#   - res 32-37: pLDDT 71-91 (confident -> very high), well-ordered coil/loop
#                feeding into TM1 -- legitimate structure, not autoinhibitory junk
#   - res 38+:   pLDDT 92+, the TM1 helix proper (UniProt's "42" call is ~4
#                residues into the actual helix, which would have truncated
#                into real secondary structure)
#
# Cutting at 32 removes only the genuinely disordered/low-confidence segment
# (including the spurious low-confidence "helix" at 11-20) while preserving
# the ordered loop (32-37) and the full native TM1 helix (38+) intact.
TM1_START = 32

print(f'\n✅ Final truncation point: residue {TM1_START} (first residue of TM1 helix)')
print(f'   Removing N-terminal residues 1–{TM1_START - 1}')

In [ ]:
#12
# Plot pLDDT confidence scores along the sequence
# Low pLDDT in N-terminus confirms disordered/flexible autoinhibitory region
from Bio import PDB

parser = PDB.PDBParser(QUIET=True)
structure = parser.get_structure('gpr151', 'gpr151_pipeline/gpr151_alphafold.pdb')

residue_ids = []
plddt_scores = []
for model in structure:
    for chain in model:
        for residue in chain:
            res_id = residue.get_id()[1]
            # In AlphaFold PDBs, B-factor column = pLDDT
            ca_atoms = [a for a in residue if a.get_name() == 'CA']
            if ca_atoms:
                residue_ids.append(res_id)
                plddt_scores.append(ca_atoms[0].get_bfactor())

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(residue_ids, plddt_scores, color='steelblue', linewidth=0.8)
ax.axvline(x=TM1_START, color='red', linestyle='--', linewidth=2, label=f'Truncation site (res {TM1_START})')
ax.axhspan(0, 70, alpha=0.25, color='r', label='Low/OK confidence (<70)')
ax.axhspan(70, 90, alpha=0.1, color='y', label='Confident (70–90)')
ax.axhspan(90, 100, alpha=0.1, color='g', label='Very confident (>90)')
ax.set_xlabel('Residue number')
ax.set_ylabel('pLDDT confidence')
ax.set_title('GPR151 AlphaFold pLDDT — low N-terminal confidence confirms flexible autoinhibitory region')
ax.legend(loc='lower right', fontsize=8)
ax.set_ylim(0, 100)
plt.tight_layout()
plt.savefig('gpr151_pipeline/plddt_plot.png', dpi=150)
plt.show()
print(f'N-terminal pLDDT (res 1–{TM1_START-1}): {np.mean(plddt_scores[:TM1_START-1]):.1f}')
print(f'TM domain pLDDT (res {TM1_START}+): {np.mean(plddt_scores[TM1_START-1:]):.1f}')

## SECTION 3: N-Terminal Truncation → GPR151-ΔN
Remove residues 1 to TM1_START-1 (the autoinhibitory N-terminal region that occludes the orthosteric pocket), keeping TM1 through TM7 and all extracellular loops intact.

In [ ]:
#14
from Bio import PDB
from Bio.PDB import PDBParser, PDBIO, Select
import warnings
warnings.filterwarnings('ignore')

class NTermTruncationSelect(Select):
    """Selects only residues >= TM1_START (removes autoinhibitory N-terminus)."""
    def __init__(self, tm1_start):
        self.tm1_start = tm1_start

    def accept_residue(self, residue):
        res_id = residue.get_id()[1]
        return res_id >= self.tm1_start

    def accept_atom(self, atom):
        # Remove OXT (terminal oxygen) from what was previously not the C-terminus
        return True

parser = PDBParser(QUIET=True)
structure = parser.get_structure('gpr151', 'gpr151_pipeline/gpr151_alphafold.pdb')

# Count residues before and after truncation
all_residues = list(structure.get_residues())
kept_residues = [r for r in all_residues if r.get_id()[1] >= TM1_START]
removed_residues = [r for r in all_residues if r.get_id()[1] < TM1_START]

print(f'Original GPR151: {len(all_residues)} residues')
print(f'Removing N-terminal residues: {len(removed_residues)} residues (1–{TM1_START-1})')
print(f'GPR151-ΔN: {len(kept_residues)} residues (starting at TM1, res {TM1_START})')

# Show what's being removed
_aa3_set = set(PDB.Polypeptide.aa3)  # tuple of standard 3-letter codes
_3to1 = PDB.Polypeptide.protein_letters_3to1  # dict, replaces removed three_to_one()
removed_seq = ''.join([_3to1[r.get_resname()]
                        for r in removed_residues
                        if r.get_resname() in _aa3_set])
print(f'\nRemoved N-terminal sequence: {removed_seq}')

# Save truncated structure
pdb_io = PDBIO()
pdb_io.set_structure(structure)
pdb_io.save('gpr151_pipeline/gpr151_delta_n.pdb', NTermTruncationSelect(TM1_START))
print(f'\n✅ Saved GPR151-ΔN to gpr151_pipeline/gpr151_delta_n.pdb')

## SECTION 4: Structure Preparation with PDBFixer
Add missing atoms, fill gaps, assign protonation states at pH 7.4, and add hydrogens.

In [ ]:
#16
from pdbfixer import PDBFixer
from openmm.app import PDBFile

print('Running PDBFixer on GPR151-ΔN...')
fixer = PDBFixer(filename='gpr151_pipeline/gpr151_delta_n.pdb')

# Find and fix missing residues (only within the kept region)
fixer.findMissingResidues()
# AlphaFold PDBs have no SEQRES records, so PDBFixer derives sequence from
# ATOM records only — our truncated structure starts at TM1_START, so PDBFixer
# sees no gap at the N-terminus. No filtering needed.
print(f'  Missing residue gaps to fill: {len(fixer.missingResidues)}')

# Handle nonstandard residues
fixer.findNonstandardResidues()
print(f'  Nonstandard residues: {fixer.nonstandardResidues}')
fixer.replaceNonstandardResidues()

# Remove heterogens (none expected in AlphaFold)
fixer.removeHeterogens(keepWater=False)

# Find and add missing heavy atoms
fixer.findMissingAtoms()
print(f'  Missing atoms to add: {sum(len(v) for v in fixer.missingAtoms.values())}')
fixer.addMissingAtoms()

# Do NOT add hydrogens here — tleap (via leaprc.protein.ff19SB) will add them
# with its own ff19SB-compatible naming. PDBFixer's addMissingHydrogens creates
# NSER (N-terminal serine cap) and HIE+HD1 atoms that have no type in ff19SB,
# causing tleap FATAL errors. tleap handles protonation correctly at pH 7.4 by
# default through its force field templates.

# Save heavy-atom-only prepared structure
with open('gpr151_pipeline/gpr151_prepared.pdb', 'w') as f:
    PDBFile.writeFile(fixer.topology, fixer.positions, f)

# Count atoms
n_atoms = fixer.topology.getNumAtoms()
n_residues = fixer.topology.getNumResidues()
print(f'  Note: hydrogens NOT added here — tleap will add them with correct ff19SB naming')
print(f'\n✅ Prepared structure: {n_residues} residues, {n_atoms} atoms (heavy atoms only)')

## SECTION 5: Membrane System Construction
Use packmol-memgen (AmberTools) to embed GPR151-ΔN in a POPC:POPE:Cholesterol bilayer (40:40:20). This places the protein through both leaflets with the extracellular loops facing outward.

In [ ]:
#18
import subprocess, os, glob

os.chdir("gpr151_pipeline")

# Root cause of the silent exit-0/no-output failures (V7, V8): packmol-memgen needs
# AMBERHOME set (normally done by sourcing /amber.sh after conda activate).
# We call the binary by full path without activating the env, so it printed
# "AMBER environment variables not found!" to stderr and bailed before packing —
# with exit code 0, which is why earlier versions saw no error and no output.
amber_env = os.environ.copy()
amber_env["AMBERHOME"] = AMBER_ENV
amber_env["PATH"] = f"{AMBER_ENV}/bin:" + amber_env.get("PATH", "")
amber_env["LD_LIBRARY_PATH"] = f"{AMBER_ENV}/lib:" + amber_env.get("LD_LIBRARY_PATH", "")

cmd = [
    PACKMOL_MEMGEN,
    "--pdb", "gpr151_prepared.pdb",
    "--lipids", "POPC:POPE:CHL1", "--ratio", "2:2:1",
    # --distxy_fix added: without it, packmol-memgen auto-computes the xy box size
    # from the protein's own geometry via pdbvol(), which crashed with
    # ZeroDivisionError: float division by zero (get_center() called on an empty
    # atom list inside pdbremix/volume.py — an internal bug for this input/version).
    # Hard-setting the xy dimension skips that broken code path entirely.
    "--distxy_fix", "65", "--dist", "25", "--dist_wat", "25",
    # --salt dropped: this packmol-memgen build (2025.1.29) errors with
    # "ERROR: The specified cation option is no available at the moment"
    # when --salt is passed (with or without --salt_c) — a build-specific
    # limitation, not a flag-naming issue. Counterions for overall charge
    # neutrality are still added automatically; we just skip the extra 0.15M NaCl.
    "--nloop", "20", "--nloop_all", "100",
    "--notprotonate",  # NOT --overwrite: memembed_align()'s "if not os.path.exists(output) and not overwrite:" is inverted, so --overwrite was silently skipping MEMEMBED. Also 'reduce' itself segfaults parsing its 66MB het dict regardless of input (confirmed via strace) -- PDBFixer already protonated at pH 7.4 in Section 4 anyway.]
]
# Diagnostic: inspect the actual pdbvol() source in THIS installed packmol-memgen
try:
    import glob as _g
    _candidates = _g.glob(f"{AMBER_ENV}/lib/python*/site-packages/packmol_memgen/main.py")
    if _candidates:
        main_py = _candidates[0]
        with open(main_py) as _f:
            _lines = _f.readlines()
        print(f"=== {main_py} lines 1760-1800 (pdbvol) ===")
        print("".join(_lines[1759:1800]))
        print("=== end pdbvol source ===\n")
    else:
        print("(packmol_memgen/main.py not found for diagnostics)")
except Exception as _e:
    print(f"(diagnostic read failed: {_e})")

print("Running (with AMBERHOME=", amber_env["AMBERHOME"], "):", " ".join(cmd))
print()
result = subprocess.run(cmd, capture_output=True, text=True, timeout=1800, env=amber_env)
print(f"=== packmol-memgen exit code: {result.returncode} ===")
print("--- STDOUT (last 3000 chars) ---")
print(result.stdout[-3000:])
print("--- STDERR (last 2000 chars) ---")
print(result.stderr[-2000:])

print("\nAll files now in gpr151_pipeline/:")
for f in sorted(glob.glob("**/*", recursive=True)):
    if os.path.isfile(f):
        print(f"  {f}: {os.path.getsize(f)/1024:.1f} KB")

In [ ]:
#19
# Identify the output files (packmol-memgen names output based on input protein name)
import glob, os

os.chdir('/kaggle/working/gpr151_pipeline')
parm7_files = glob.glob('*.parm7') + glob.glob('*.top') + glob.glob('*.prmtop')
rst7_files = glob.glob('*.rst7') + glob.glob('*.inpcrd') + glob.glob('*.crd')

if parm7_files and rst7_files:
    PARM7 = parm7_files[0]
    RST7 = rst7_files[0]
    MEMBRANE_PDB = None  # signal to cell 24 to use PARM7/RST7 directly
    print(f'✅ Membrane system files:')
    print(f'   Topology: {PARM7}')
    print(f'   Coordinates: {RST7}')
else:
    # Check for PDB output — packmol-memgen names it after the input PDB
    pdb_files = (glob.glob('*bilayer*.pdb') + glob.glob('*membrane*.pdb') +
                 glob.glob('gpr151_prepared*.pdb') + glob.glob('bilayer*.pdb'))
    # Exclude the input prepared PDB itself
    pdb_files = [f for f in pdb_files if 'delta_n' not in f and f != 'gpr151_prepared.pdb']
    if not pdb_files:
        # Last resort: any PDB larger than the input (the bilayer will be huge)
        all_pdbs = glob.glob('*.pdb')
        input_size = os.path.getsize('gpr151_prepared.pdb') if os.path.exists('gpr151_prepared.pdb') else 0
        pdb_files = [f for f in all_pdbs if os.path.getsize(f) > input_size]
    if pdb_files:
        MEMBRANE_PDB = sorted(pdb_files, key=os.path.getsize, reverse=True)[0]  # largest = bilayer
        print(f'✅ Membrane PDB: {MEMBRANE_PDB}')
    else:
        print('⚠️  packmol-memgen output not found — check output above for errors')
        print('Listing all files:')
        for f in sorted(os.listdir('.')):
            print(f'  {f}: {os.path.getsize(f)/1024:.1f} KB')
        os.chdir('..')
        raise RuntimeError('packmol-memgen did not produce a membrane PDB/topology -- see the packmol-memgen stdout/stderr printed above for the real error before re-running this cell.')

os.chdir('..')

## SECTION 6: OpenMM Simulation Setup

Load the membrane system via **ParmEd** (reads the CHARMM-style whole-residue names
`POPC`/`POPE`/`CHL1`/`TIP3` that packmol-memgen writes — `tleap` truncates these to 3 characters
and fails with `Unknown residue: POP`), build the OpenMM `System` with the native **CHARMM36m**
force field (already defines these residues, no AMBER lipid parameters needed), configure the
membrane-specific NPT barostat, and set up the Langevin integrator with hydrogen mass
repartitioning for a 4 fs timestep.

This is the fix verified against the actual bilayer output during the Colab debugging session,
ported unchanged — the bug it works around (tleap + CHARMM-style residue names) is independent
of which notebook platform runs it.


In [ ]:
#21
import subprocess, os
# Ensure we're in /kaggle/working
os.chdir('/kaggle/working')
from openmm.app import PME, HBonds, Simulation, AmberPrmtopFile, AmberInpcrdFile
from openmm import Platform
from openmm.unit import nanometer, nanometers, amu, bar, kelvin, picosecond, picoseconds, kilojoules_per_mole
from openmm import MonteCarloMembraneBarostat, LangevinMiddleIntegrator

# Two paths depending on what packmol-memgen produced:
#   MEMBRANE_PDB = None  → got parm7+rst7 directly (load without tleap)
#   MEMBRANE_PDB = str   → got a PDB (need tleap to build prmtop/inpcrd)

if MEMBRANE_PDB is None:
    # packmol-memgen gave us AMBER topology directly — load it
    print(f'Loading membrane system from AMBER topology: {PARM7} + {RST7}')
    prmtop = AmberPrmtopFile(f'gpr151_pipeline/{PARM7}')
    inpcrd = AmberInpcrdFile(f'gpr151_pipeline/{RST7}')
else:
    BILAYER_PDB = f'gpr151_pipeline/{MEMBRANE_PDB}'
    print(f'Building system via tleap (AMBER ff19SB + Lipid21): {BILAYER_PDB}')

    # --- FIX: Run pdb4amber before tleap to convert CHARMM lipid names ---
    env = dict(os.environ)
    env['AMBERHOME'] = '/opt/miniconda/envs/ambertools'
    env['PATH'] = f"/opt/miniconda/envs/ambertools/bin:" + env.get("PATH", "")
    subprocess.run([
        '/opt/miniconda/envs/ambertools/bin/pdb4amber',
        BILAYER_PDB,
        '-o', 'gpr151_pipeline/bilayer_clean.pdb'
    ], env=env, check=True)
    print('pdb4amber completed')
    BILAYER_PDB = 'gpr151_pipeline/bilayer_clean.pdb'
    # --- END FIX ---

    with open(BILAYER_PDB) as f:
        _lines = f.readlines()
    _cryst = [l for l in _lines if l.startswith('CRYST1')]
    if _cryst:
        box_a, box_b, box_c = (float(x) for x in _cryst[0].split()[1:4])
    else:
        print('  No CRYST1 record found -- computing box from atom coordinates')
        _xs, _ys, _zs = [], [], []
        for l in _lines:
            if l.startswith('ATOM') or l.startswith('HETATM'):
                _xs.append(float(l[30:38]))
                _ys.append(float(l[38:46]))
                _zs.append(float(l[46:54]))
        box_a = max(_xs) - min(_xs)
        box_b = max(_ys) - min(_ys)
        box_c = max(_zs) - min(_zs)
    print(f'  Box: {box_a:.2f} x {box_b:.2f} x {box_c:.2f} A')

    leap_script = f"""source leaprc.protein.ff19SB
source leaprc.water.tip3p
source leaprc.lipid21
mol = loadpdb {BILAYER_PDB}
set mol box {{ {box_a} {box_b} {box_c} }}
saveamberparm mol gpr151_pipeline/system.prmtop gpr151_pipeline/system.inpcrd
quit
"""
    with open('gpr151_pipeline/build.leap', 'w') as f:
        f.write(leap_script)

    env = dict(os.environ)
    env['AMBERHOME'] = '/opt/miniconda/envs/ambertools'
    result = subprocess.run(
        ['/opt/miniconda/envs/ambertools/bin/tleap', '-f', 'gpr151_pipeline/build.leap'],
        capture_output=True, text=True, env=env,
    )
    print(result.stdout[-2000:])
    if result.returncode != 0 or not os.path.exists('gpr151_pipeline/system.prmtop'):
        raise RuntimeError(f'tleap failed:\n{result.stdout}\n{result.stderr}')

    prmtop = AmberPrmtopFile('gpr151_pipeline/system.prmtop')
    inpcrd = AmberInpcrdFile('gpr151_pipeline/system.inpcrd')
topology = prmtop.topology
positions = inpcrd.positions
box_vectors = inpcrd.boxVectors

print('Creating OpenMM system...')
system = prmtop.createSystem(
    nonbondedMethod=PME, nonbondedCutoff=1.2 * nanometer,
    constraints=HBonds, hydrogenMass=1.5 * amu, rigidWater=True, removeCMMotion=True,
)
print(f'  System: {system.getNumParticles():,} particles')

barostat = MonteCarloMembraneBarostat(
    1.0 * bar, 0.0 * bar * nanometer, 310.0 * kelvin,
    MonteCarloMembraneBarostat.XYIsotropic, MonteCarloMembraneBarostat.ZFree, 25,
)
system.addForce(barostat)

integrator = LangevinMiddleIntegrator(310.0 * kelvin, 1.0 / picosecond, 0.004 * picosecond)
integrator.setRandomNumberSeed(42)

try:
    platform = Platform.getPlatformByName('CUDA')
    properties = {'CudaPrecision': 'mixed'}
    print('Using CUDA GPU platform')
except Exception:
    try:
        platform = Platform.getPlatformByName('OpenCL')
        properties = {}
        print('Using OpenCL platform')
    except Exception:
        platform = Platform.getPlatformByName('CPU')
        properties = {}
        print('WARNING: CPU platform — check accelerator setting in Kaggle Settings sidebar')

simulation = Simulation(topology, system, integrator, platform, properties)
simulation.context.setPositions(positions)
if box_vectors is not None:
    simulation.context.setPeriodicBoxVectors(*box_vectors)

print(f'\n✅ OpenMM simulation created: {system.getNumParticles():,} particles')

## SECTION 7: Energy Minimization + Equilibration
Minimization removes steric clashes from the membrane builder. Equilibration runs in stages with progressively released restraints on protein heavy atoms.

In [ ]:
#23
import time
from openmm.app import PDBFile
from openmm import CustomExternalForce

# --- Add protein restraints BEFORE creating the simulation ---
restraint = CustomExternalForce('k*periodicdistance(x, y, z, x0, y0, z0)^2')
restraint.addGlobalParameter('k', 1000.0*kilojoules_per_mole/nanometer**2)
restraint.addPerParticleParameter('x0')
restraint.addPerParticleParameter('y0')
restraint.addPerParticleParameter('z0')

# Protein-positive filter (same as your cell 24)
protein_residues = {'ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY',
                    'HIS','HIE','HID','HIP','ILE','LEU','LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL',
                    'NALA','NASP','NASN','NGLU','NGLN','NSER','NTHR','NLEU','NILE',
                    'NVAL','NPHE','NTRP','NTYR','NCYS','NMET','NARG','NLYS','NHIS',
                    'NHIE','NHID','NPRO','NGLY','CYX','CYM'}

n_restrained = 0
for atom in topology.atoms():
    if atom.residue.name in protein_residues and atom.element.symbol != 'H':
        restraint.addParticle(atom.index, positions[atom.index].value_in_unit(nanometer))
        n_restrained += 1

system.addForce(restraint)
print(f'Added restraints on {n_restrained} protein heavy atoms (expected ~3,000–3,500)')

# --- Recreate simulation with restraints ---
integrator = LangevinMiddleIntegrator(310.0*kelvin, 1.0/picosecond, 0.004*picosecond)
simulation = Simulation(topology, system, integrator, platform, properties)
simulation.context.setPositions(positions)
if box_vectors is not None:
    simulation.context.setPeriodicBoxVectors(*box_vectors)

initial_energy = simulation.context.getState(getEnergy=True).getPotentialEnergy()
print(f'Initial energy: {initial_energy.value_in_unit(kilojoules_per_mole):.1f} kJ/mol')

# --- Phase 1: Minimize with strong restraints ---
print('Phase 1: Minimizing with protein restrained...')
simulation.minimizeEnergy(tolerance=10.0*kilojoules_per_mole/nanometer, maxIterations=10000)

# --- Phase 2: Reduce restraint strength ---
simulation.context.setParameter('k', 100.0*kilojoules_per_mole/nanometer**2)
print('Phase 2: Weak restraints...')
simulation.minimizeEnergy(tolerance=10.0*kilojoules_per_mole/nanometer, maxIterations=10000)

# --- Phase 3: Remove restraints ---
simulation.context.setParameter('k', 0.0)
print('Phase 3: Final unrestrained minimization...')
simulation.minimizeEnergy(tolerance=10.0*kilojoules_per_mole/nanometer, maxIterations=10000)

final_energy = simulation.context.getState(getEnergy=True).getPotentialEnergy()
print(f'  Final energy: {final_energy.value_in_unit(kilojoules_per_mole):.1f} kJ/mol')

# Save minimized structure
state = simulation.context.getState(getPositions=True)
with open('gpr151_pipeline/gpr151_minimized.pdb', 'w') as f:
    PDBFile.writeFile(simulation.topology, state.getPositions(), f)
print('  Saved: gpr151_minimized.pdb')

In [ ]:
#24
# --- Equilibration Phase 1: NVT with protein restraints (1 ns) ---
# Apply positional restraints to protein heavy atoms
from openmm import CustomExternalForce

def add_protein_restraints(system, topology, positions, k=1000.0):
    """Add flat-bottom positional restraints to protein heavy atoms."""
    restraint = CustomExternalForce('k*((x-x0)^2+(y-y0)^2+(z-z0)^2)')
    restraint.addGlobalParameter('k', k * kilojoules_per_mole/nanometer**2)
    restraint.addPerParticleParameter('x0')
    restraint.addPerParticleParameter('y0')
    restraint.addPerParticleParameter('z0')

    protein_residues = {'ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY',
                        'HIS','HIE','HID','HIP',  # HIS variants in AMBER ff19SB
                        'ILE','LEU','LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL',
                        'NALA','NASP','NASN','NGLU','NGLN','NSER','NTHR','NLEU','NILE',
                        'NVAL','NPHE','NTRP','NTYR','NCYS','NMET','NARG','NLYS','NHIS',
                        'NHIE','NHID','NPRO','NGLY'}
    n_restrained = 0
    for atom in topology.atoms():
        if atom.residue.name in protein_residues and atom.element.symbol != 'H':
            pos = positions[atom.index].value_in_unit(nanometers)
            restraint.addParticle(atom.index, pos)
            n_restrained += 1
    system.addForce(restraint)
    return n_restrained

state = simulation.context.getState(getPositions=True)
n_restrained = add_protein_restraints(system, topology, state.getPositions())
simulation.context.reinitialize(preserveState=True)
print(f'Applied restraints to {n_restrained} protein heavy atoms')

# Set initial velocities
simulation.context.setVelocitiesToTemperature(310*kelvin, 42)

# Reporters
from openmm.app import StateDataReporter, DCDReporter

simulation.reporters.append(StateDataReporter(
    'gpr151_pipeline/equilibration.log', 2500,  # every 10 ps
    step=True, time=True, potentialEnergy=True, kineticEnergy=True,
    totalEnergy=True, temperature=True, volume=True, progress=True,
    remainingTime=True, speed=True, totalSteps=250000))

print('\nStep 2: NVT equilibration with protein restraints (1 ns)...')
t0 = time.time()
simulation.step(250000)  # 1 ns at 4 fs/step
print(f'  Done in {time.time()-t0:.0f}s')

state = simulation.context.getState(getPositions=True)
with open('gpr151_pipeline/gpr151_nvt.pdb', 'w') as f:
    PDBFile.writeFile(simulation.topology, state.getPositions(), f)
print('  Saved: gpr151_nvt.pdb')

In [ ]:
#25
# --- Equilibration Phase 2: NPT with reduced restraints (2 ns) ---
# Remove restraints from the system (replace with weaker ones)
# Re-create system without restraints but with barostat
system2 = prmtop.createSystem(
    nonbondedMethod=PME,
    nonbondedCutoff=1.2*nanometers,
    constraints=HBonds,
    hydrogenMass=1.5*amu
)
system2.addForce(MonteCarloMembraneBarostat(
    1.0*bar, 0.0*bar*nanometer, 310.0*kelvin,
    MonteCarloMembraneBarostat.XYIsotropic, MonteCarloMembraneBarostat.ZFree, 25))

# Add weak (50 kJ/mol/nm²) restraints for NPT equilibration
state = simulation.context.getState(getPositions=True, getVelocities=True)
n_restrained = add_protein_restraints(system2, topology, state.getPositions(), k=50.0)

integrator2 = LangevinMiddleIntegrator(310.0*kelvin, 1.0/picosecond, 0.004*picoseconds)
integrator2.setRandomNumberSeed(43)

simulation2 = Simulation(topology, system2, integrator2, platform, properties)
simulation2.context.setPositions(state.getPositions())
simulation2.context.setVelocities(state.getVelocities())
if state.getPeriodicBoxVectors() is not None:
    simulation2.context.setPeriodicBoxVectors(*state.getPeriodicBoxVectors())

simulation2.reporters.append(StateDataReporter(
    'gpr151_pipeline/npt_equil.log', 2500,
    step=True, time=True, potentialEnergy=True, temperature=True,
    volume=True, density=True, progress=True, remainingTime=True,
    speed=True, totalSteps=500000))

print('Step 3: NPT equilibration with weak restraints (2 ns)...')
t0 = time.time()
simulation2.step(500000)  # 2 ns
print(f'  Done in {time.time()-t0:.0f}s')

state2 = simulation2.context.getState(getPositions=True, getVelocities=True)
with open('gpr151_pipeline/gpr151_npt.pdb', 'w') as f:
    PDBFile.writeFile(simulation2.topology, state2.getPositions(), f)
print('  Saved: gpr151_npt.pdb')

# ADD THIS AT THE END OF CELL 25
simulation2.saveCheckpoint('/kaggle/working/gpr151_npt.checkpoint')
print('  Saved NPT checkpoint: gpr151_npt.checkpoint')

## SECTION 8: Production MD — Checkpoint-Resume (Kaggle Datasets)

Same design as the Colab version's Google-Drive checkpointing — 5 ns chunks (`CHUNK_STEPS =
1,250,000`), same file names (`checkpoint.chk`, `chunk_NNN.dcd`, `production.log`,
`steps_completed.txt`) — just persisted via a Kaggle Dataset across sessions instead of a Drive
folder (see SECTION 0.5 for the resume procedure). Within a single ~9h session this just
accumulates chunks in `/kaggle/working/gpr151_pipeline/` directly; only carrying state *between*
sessions needs the publish-as-dataset step.


In [ ]:
#27
import os, time, glob, shutil

WORK_DIR = 'gpr151_pipeline'  # was DRIVE_DIR in the Colab version — Kaggle persists this via
                               # Datasets across sessions instead of via a Drive mount
os.makedirs(WORK_DIR, exist_ok=True)

# Production MD settings — identical to the Colab version
PROD_NS      = 20           # total nanoseconds to simulate
STEPS_PER_NS = 250_000      # 4 fs timestep -> 250k steps/ns
TOTAL_STEPS  = PROD_NS * STEPS_PER_NS
SAVE_EVERY   = 25_000       # trajectory frame every 100 ps
CHUNK_STEPS  = 1_250_000    # 5 ns per chunk (safe within a ~9h Kaggle session)

progress_file   = f'{WORK_DIR}/steps_completed.txt'
checkpoint_file = f'{WORK_DIR}/checkpoint.chk'

steps_done = 0
if os.path.exists(progress_file):
    with open(progress_file) as f:
        steps_done = int(f.read().strip())
    print(f'Resuming: {steps_done/STEPS_PER_NS:.1f} ns done '
          f'({steps_done:,}/{TOTAL_STEPS:,} steps)')
else:
    print('Fresh start — no prior checkpoint found')

if steps_done >= TOTAL_STEPS:
    print(f'\n✅ Production MD already complete ({PROD_NS} ns). Skip to Section 9.')
else:
    ns_left = (TOTAL_STEPS - steps_done) / STEPS_PER_NS
    print(f'Remaining: {ns_left:.1f} ns')


In [ ]:
#28
import os, time
from openmm.app import (AmberPrmtopFile, AmberInpcrdFile, PME, HBonds, Simulation,
                        DCDReporter, StateDataReporter, CheckpointReporter, PDBFile)
from openmm import Platform, MonteCarloMembraneBarostat, LangevinMiddleIntegrator
from openmm.unit import (nanometer, nanometers, amu, bar, kelvin, picosecond, 
                         picoseconds, kilojoules_per_mole)

# --- 1. Load system files ---
os.chdir('/kaggle/working/gpr151_pipeline')
prmtop = AmberPrmtopFile('system.prmtop')
inpcrd = AmberInpcrdFile('system.inpcrd')

topology = prmtop.topology
positions = inpcrd.positions
box_vectors = inpcrd.boxVectors

print(f"Loaded: {topology.getNumAtoms()} atoms")

# --- 2. Build system ---
system = prmtop.createSystem(
    nonbondedMethod=PME,
    nonbondedCutoff=1.2*nanometers,
    constraints=HBonds,
    hydrogenMass=1.5*amu,
    rigidWater=True,
    removeCMMotion=True,
)

system.addForce(MonteCarloMembraneBarostat(
    1.0*bar, 0.0*bar*nanometer, 310.0*kelvin,
    MonteCarloMembraneBarostat.XYIsotropic,
    MonteCarloMembraneBarostat.ZFree, 25))

# --- 3. Integrator & platform ---
integrator = LangevinMiddleIntegrator(310.0*kelvin, 1.0/picosecond, 0.004*picoseconds)
integrator.setRandomNumberSeed(44)

try:
    platform = Platform.getPlatformByName('CUDA')
    properties = {'CudaPrecision': 'mixed'}
    print('CUDA platform')
except Exception:
    platform = Platform.getPlatformByName('OpenCL')
    properties = {}
    print('OpenCL platform')

# --- 4. Create simulation ---
sim_prod = Simulation(topology, system, integrator, platform, properties)

# --- 5. Resume or initialize (CORRECT ORDER) ---
production_chk = '/kaggle/working/gpr151_md.checkpoint'
npt_chk = '/kaggle/working/gpr151_npt.checkpoint'
npt_offset_file = '/kaggle/working/gpr151_pipeline/npt_offset.txt'
steps_done = 0
npt_offset = 0

if os.path.exists(npt_offset_file):
    with open(npt_offset_file) as f:
        npt_offset = int(f.read().strip())

if os.path.exists(npt_chk):
    sim_prod.loadCheckpoint(npt_chk)
    if npt_offset == 0:
        npt_offset = sim_prod.context.getState().getStepCount()
        with open(npt_offset_file, 'w') as f:
            f.write(str(npt_offset))
    steps_done = 0
    print('Loaded NPT checkpoint (3 ns equilibrated) — starting production from here')
elif os.path.exists(production_chk):
    sim_prod.loadCheckpoint(production_chk)
    steps_done = sim_prod.context.getState().getStepCount() - npt_offset
    print(f'Resumed production checkpoint at {steps_done/250000:.1f} ns')
elif os.path.exists('/kaggle/working/gpr151_npt.pdb'):
    from openmm.app import PDBFile
    npt_state = PDBFile('/kaggle/working/gpr151_npt.pdb')
    sim_prod.context.setPositions(npt_state.positions)
    sim_prod.context.setVelocitiesToTemperature(310 * kelvin)
    if box_vectors is not None:
        sim_prod.context.setPeriodicBoxVectors(*box_vectors)
    print('Loaded NPT PDB + randomized velocities')
else:
    sim_prod.context.setPositions(positions)
    sim_prod.context.setPeriodicBoxVectors(*box_vectors)
    print('WARNING: Loaded raw inpcrd (completely unequilibrated)')

# --- 6. Reporters (FIXED PATHS & APPEND LOGIC) ---
os.chdir('/kaggle/working')
dcd_path = os.path.join('/kaggle/working', 'gpr151_md.dcd')
log_path = os.path.join('/kaggle/working', 'gpr151_md.log')
chk_path = os.path.join('/kaggle/working', 'gpr151_md.checkpoint')

# Only append if DCD already exists (fresh run = False)
dcd_append = os.path.exists(dcd_path)

sim_prod.reporters = []
sim_prod.reporters.append(DCDReporter(dcd_path, 25000, append=dcd_append))
sim_prod.reporters.append(StateDataReporter(log_path, 25000,
    step=True, time=True, potentialEnergy=True, temperature=True,
    volume=True, speed=True, remainingTime=True, append=dcd_append,
    totalSteps=TOTAL_STEPS))
sim_prod.reporters.append(CheckpointReporter(chk_path, 25000))
print(f'Reporters attached (DCD append={dcd_append})')

# --- 7. Production run ---
TOTAL_STEPS = 5000000   # 20 ns
steps_per_ns = 250000

if steps_done >= TOTAL_STEPS:
    print(f'Already complete: {steps_done/steps_per_ns:.1f} ns')
else:
    remaining = TOTAL_STEPS - steps_done
    print(f'Running {remaining/steps_per_ns:.1f} ns ({remaining:,} steps)')

    for chunk_start in range(steps_done, TOTAL_STEPS, steps_per_ns * 5):
        chunk_end = min(chunk_start + steps_per_ns * 5, TOTAL_STEPS)
        chunk_steps = chunk_end - chunk_start

        print(f"\nChunk: {chunk_start/steps_per_ns:.1f} -> {chunk_end/steps_per_ns:.1f} ns ({chunk_steps:,} steps)")

        t0 = time.time()
        sim_prod.step(chunk_steps)
        elapsed = time.time() - t0

        speed = chunk_steps / elapsed / 1000
        print(f"  Done in {elapsed/60:.1f} min ({speed:.1f} ns/day)")
        
        # Save progress for cell 27 compatibility
        with open('gpr151_pipeline/steps_completed.txt', 'w') as f:
            f.write(str(sim_prod.context.getState().getStepCount() - npt_offset))

    print(f"\n{'='*50}")
    print(f"Production complete")

In [ ]:
#29
import glob, os, MDAnalysis as mda
from MDAnalysis import Writer

# Cell 28 saves a single DCD, not chunks
dcd_path = '/kaggle/working/gpr151_md.dcd'
pdb_path = 'gpr151_pipeline/gpr151_npt.pdb'

if os.path.exists(dcd_path):
    u = mda.Universe(pdb_path, dcd_path)
    n_frames = len(u.trajectory)
    total_ns = n_frames * 0.1  # 100 ps per frame
    print(f'Production trajectory: {n_frames} frames ({total_ns:.1f} ns)')
    print(f'Ready for Section 9 (pocket analysis)')
else:
    print('No production trajectory found. Run Section 8 first.')

## SECTION 9: Orthosteric Pocket Analysis
Analyze the MD trajectory to characterize the orthosteric pocket: volume, accessibility, and geometry. Identify the top-N snapshots with the most open/accessible binding site for ensemble docking.

In [ ]:
#30b
# Guard cell: verify Section 8 outputs exist before running analysis
import os

required_files = {
    'NPT structure': 'gpr151_pipeline/gpr151_npt.pdb',
    'Production DCD': '/kaggle/working/gpr151_md.dcd',
    'Minimized structure': 'gpr151_pipeline/gpr151_minimized.pdb',
}

missing = []
for label, path in required_files.items():
    if not os.path.exists(path):
        missing.append(f'  {label}: {path}')

if missing:
    print('❌ Missing required files — run Sections 7–8 first:')
    print('\n'.join(missing))
    raise FileNotFoundError('Cannot run Section 9 without Section 8 outputs.')
else:
    print('✅ All required files present. Ready for Section 9.')

In [ ]:
#31
import MDAnalysis as mda
from MDAnalysis.analysis import rms
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import ConvexHull

# Load trajectory
print('Loading MD trajectory...')
u = mda.Universe('gpr151_pipeline/gpr151_npt.pdb',
                 '/kaggle/working/gpr151_md.dcd')

print(f'Trajectory: {len(u.trajectory)} frames')
print(f'Atoms: {len(u.atoms)}')

# Select protein atoms only
protein = u.select_atoms('protein')
print(f'Protein atoms: {len(protein)}')

# Select backbone for RMSD alignment
backbone = u.select_atoms('protein and backbone')

# --- RMSD analysis ---
# RMSD: align to backbone, measure backbone RMSD
# MDAnalysis 2.x compatible signature
R = rms.RMSD(backbone, backbone, select='backbone')
R.run()

rmsd_data = R.results.rmsd
times = rmsd_data[:, 1]    # time in ps
rmsd_vals = rmsd_data[:, 2]  # RMSD in Angstroms

fig, axes = plt.subplots(2, 1, figsize=(12, 6))
axes[0].plot(times/1000, rmsd_vals, color='navy', lw=0.8)
axes[0].set_xlabel('Time (ns)')
axes[0].set_ylabel('RMSD (Å)')
axes[0].set_title('GPR151-ΔN backbone RMSD over production MD')
axes[0].axhline(y=np.mean(rmsd_vals), color='red', ls='--', lw=1, label=f'Mean: {np.mean(rmsd_vals):.2f} Å')
axes[0].legend()

# --- Estimate orthosteric pocket volume ---
# The orthosteric pocket of Class A GPCRs is between TM3, TM5, TM6, TM7
# Select residues in the canonical binding pocket region
# Using GPCRDB generic positions 3.32, 3.33, 3.36, 5.42, 5.43, 6.48, 6.51, 6.52, 7.39, 7.42, 7.43
# For now, select all residues within 10 Å of the pocket centroid

# Define pocket centroid (center of TM bundle, extracellular half)
# We'll approximate this as the centroid of TM3/5/6/7 Cα atoms in the extracellular region
# This will be refined after the first frame analysis

pocket_volumes = []
frame_indices = []

print('\nAnalyzing pocket volume across trajectory...')
# Sample every 5th frame for speed
for ts in u.trajectory[::5]:
    # Select protein Cα atoms roughly in the extracellular-facing TM region
    # (upper 1/3 of the 7TM bundle)
    ca_atoms = protein.select_atoms('name CA')
    ca_pos = ca_atoms.positions

    # Get z-range of protein, focus on upper extracellular third
    z_min, z_max = ca_pos[:, 2].min(), ca_pos[:, 2].max()
    z_ec_cutoff = z_min + (z_max - z_min) * 0.5  # upper half

    # Select atoms in pocket region (upper half of TM bundle)
    pocket_region = protein.select_atoms(
        f'(name CA or name CB or name CG) and prop z > {z_ec_cutoff}')

    if len(pocket_region) > 4:
        try:
            hull = ConvexHull(pocket_region.positions)
            pocket_volumes.append(hull.volume)  # in Å³
        except Exception:
            pocket_volumes.append(0)
    else:
        pocket_volumes.append(0)
    frame_indices.append(ts.frame)

pocket_volumes = np.array(pocket_volumes)
frame_indices = np.array(frame_indices)

axes[1].plot(np.array(frame_indices) * 0.1, pocket_volumes, color='darkorange', lw=0.8)
axes[1].set_xlabel('Time (ns)')
axes[1].set_ylabel('Approximate pocket volume (Å³)')
axes[1].set_title('Orthosteric pocket accessibility over MD trajectory')

plt.tight_layout()
plt.savefig('gpr151_pipeline/md_analysis.png', dpi=150)
plt.show()

print(f'Mean pocket volume: {np.mean(pocket_volumes):.0f} Å³')
print(f'Max pocket volume: {np.max(pocket_volumes):.0f} Å³ at frame {frame_indices[np.argmax(pocket_volumes)]}')

In [ ]:
#32
import os  # ensure os is available
# Extract top-20 most open frames for ensemble docking
N_DOCKING_FRAMES = 20

# Only use frames from the second half of the trajectory (equilibrated)
n_total = len(frame_indices)
equilibrated_mask = frame_indices >= frame_indices[n_total // 2]

eq_volumes = pocket_volumes[equilibrated_mask]
eq_frames = frame_indices[equilibrated_mask]

# Select top-N frames by pocket volume, ensuring spread across trajectory
sorted_idx = np.argsort(eq_volumes)[::-1]
selected_frames = []
MIN_FRAME_SEPARATION = max(1, len(eq_frames) // (N_DOCKING_FRAMES * 2))

for idx in sorted_idx:
    frame = eq_frames[idx]
    if all(abs(frame - sf) >= MIN_FRAME_SEPARATION for sf in selected_frames):
        selected_frames.append(frame)
    if len(selected_frames) >= N_DOCKING_FRAMES:
        break

if len(selected_frames) == 0:
    print('WARNING: No frames selected for docking. Using frame 0.')
    selected_frames = [0]

print(f'Selected {len(selected_frames)} frames for ensemble docking:')
print(f'Frames: {sorted(selected_frames)}')

# Export selected frames as PDB files for docking
os.makedirs('gpr151_pipeline/docking_frames', exist_ok=True)

for frame_num in selected_frames:
    u.trajectory[frame_num]
    protein_only = u.select_atoms('protein')
    with mda.Writer(f'gpr151_pipeline/docking_frames/frame_{frame_num:04d}.pdb',
                    protein_only.n_atoms) as W:
        W.write(protein_only)

print(f'✅ Exported {len(selected_frames)} receptor conformations for ensemble docking')

## SECTION 10: AutoDock Vina — Docking GUM-Class Psoralen Compounds
Dock psoralen scaffold and GUM-class analogs into the ensemble of open receptor conformations. The psoralen SMILES are the parent scaffold and several substituted analogs — replace with exact GUM3/GUM4 SMILES from Yoshida et al. 2026 (Genes to Cells, DOI: 10.1111/gtc.70119).

In [ ]:
#34
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, Draw
from rdkit.Chem.Draw import IPythonConsole
import subprocess

# GUM-class psoralen compound library
# NOTE: Replace GUM3_SMILES and GUM4_SMILES with actual SMILES from Yoshida 2026
# All have psoralen ring system (6H-furo[3,2-g][1]benzopyran-6-one)
GUM_LIBRARY = {
    'Psoralen':          'O=c1ccc2cc3ccoc3cc2o1',
    '8-MOP (Methoxsalen)': 'COc1c2ccoc2cc2oc(=O)ccc12',
    '5-MOP (Bergapten)': 'COc1ccc2ccc(=O)oc2c1-c1ccoc1',
    'Isopsoralen':       'O=c1ccc2cc3occc3cc2o1',
    'Xanthotoxol':       'OCc1ccoc1-c1cc2ccc(=O)oc2cc1O',
    'GUM3': 'COc1ccccc1CNC(=O)CC2=C(C)c3cc4c(C)coc4c(C)c3OC2=O',
    'GUM4': 'Cc1oc2cc3OC(=O)C(=C(C)c3cc2c1C)CC(=O)NCCc4ccc(Cl)cc4',
}

print('GUM-class compound library:')
valid_compounds = {}
for name, smiles in GUM_LIBRARY.items():
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        mw = Descriptors.ExactMolWt(mol)
        logp = Descriptors.MolLogP(mol)
        tpsa = Descriptors.TPSA(mol)
        hbd = Descriptors.NumHDonors(mol)
        hba = Descriptors.NumHAcceptors(mol)
        print(f'  {name}: MW={mw:.1f}, LogP={logp:.2f}, TPSA={tpsa:.1f}, HBD={hbd}, HBA={hba}')
        valid_compounds[name] = (smiles, mol)
    else:
        print(f'  {name}: INVALID SMILES')

print(f'\n{len(valid_compounds)} valid compounds for docking')

In [ ]:
#35
import subprocess

import shutil
assert shutil.which('obabel'), 'Open Babel (obabel) not found. Install with: apt-get install openbabel'

import os

os.makedirs('gpr151_pipeline/ligands', exist_ok=True)
os.makedirs('gpr151_pipeline/docking_results', exist_ok=True)

def prepare_ligand(name, smiles, output_dir):
    """Generate 3D conformation and convert to PDBQT for AutoDock Vina."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        print(f'  {name}: RDKit could not parse SMILES')
        return None
    mol = Chem.AddHs(mol)
    result = AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())
    if result == -1:
        # ETKDGv3 failed; try with random coords
        AllChem.EmbedMolecule(mol, AllChem.ETKDGv3(), randomSeed=0xf00d)
    AllChem.MMFFOptimizeMolecule(mol)

    # Save as SDF
    sdf_path = f'{output_dir}/{name.replace(" ","_")}.sdf'
    with Chem.SDWriter(sdf_path) as w:
        w.write(mol)

    # Convert SDF → PDBQT using Open Babel
    pdbqt_path = f'{output_dir}/{name.replace(" ","_")}.pdbqt'
    result = subprocess.run(
        ['obabel', sdf_path, '-O', pdbqt_path, '-h', '--partialcharge', 'gasteiger'],
        capture_output=True, text=True)
    return pdbqt_path if os.path.exists(pdbqt_path) else None

def prepare_receptor(pdb_path, output_path):
    """Convert receptor PDB to PDBQT for AutoDock Vina."""
    result = subprocess.run(
        ['obabel', pdb_path, '-O', output_path, '-xr'],
        capture_output=True, text=True)
    return output_path if os.path.exists(output_path) else None

print('Preparing ligands...')
ligand_pdbqts = {}
for name, (smiles, mol) in valid_compounds.items():
    path = prepare_ligand(name, smiles, 'gpr151_pipeline/ligands')
    if path:
        ligand_pdbqts[name] = path
        print(f'  ✓ {name}')
    else:
        print(f'  ✗ {name} — preparation failed')

print(f'\n{len(ligand_pdbqts)} ligands prepared')

In [ ]:
#36
# Determine docking box center from pocket analysis
# Use the frame with the largest pocket volume
best_frame = frame_indices[np.argmax(pocket_volumes)]
u.trajectory[best_frame]

# Compute pocket centroid as center of TM bundle upper region
protein_atoms = u.select_atoms('protein')
ca_pos = u.select_atoms('protein and name CA').positions
z_min, z_max = ca_pos[:, 2].min(), ca_pos[:, 2].max()
z_upper = z_min + (z_max - z_min) * 0.55  # slightly above center

pocket_atoms = u.select_atoms(f'protein and name CA and prop z > {z_upper}')
if len(pocket_atoms) == 0:
    # Fallback: use all CA atoms centroid
    pocket_atoms = u.select_atoms('protein and name CA')
pocket_center = pocket_atoms.positions.mean(axis=0)

BOX_CENTER = tuple(pocket_center)  # (x, y, z) in Angstroms
BOX_SIZE = (25.0, 25.0, 25.0)  # 25×25×25 Å box

print(f'Docking box center: ({BOX_CENTER[0]:.2f}, {BOX_CENTER[1]:.2f}, {BOX_CENTER[2]:.2f}) Å')
print(f'Docking box size: {BOX_SIZE} Å')

In [ ]:
#37
from vina import Vina
import os, json

all_docking_results = {}

# Convert to plain Python floats (Vina's SWIG bindings reject numpy/Quantity)
cx = float(BOX_CENTER[0])
cy = float(BOX_CENTER[1])
cz = float(BOX_CENTER[2])
sx = float(BOX_SIZE[0])
sy = float(BOX_SIZE[1])
sz = float(BOX_SIZE[2])

print(f'Docking box center: ({cx}, {cy}, {cz}) Å')
print(f'Docking box size: ({sx}, {sy}, {sz}) Å')

for ligand_name, ligand_pdbqt in ligand_pdbqts:
    print(f'\nDocking {ligand_name}...')
    results = []
    
    for frame_idx in selected_frames:
        receptor = f'frame_{frame_idx:04d}.pdbqt'
        
        if not os.path.exists(receptor):
            print(f'  Frame {frame_idx}: receptor not found')
            continue
        
        try:
            v = Vina(sf_name='vina', cpu=4)
            v.set_receptor(receptor)
            v.set_ligand_from_file(ligand_pdbqt)
            
            # Pass 6 individual floats, not keyword args
            v.compute_vina_maps(cx, cy, cz, sx, sy, sz)
            
            v.dock(exhaustiveness=32)
            energies = v.energies()
            
            if energies is not None and len(energies) > 0:
                best = float(energies[0][0])
                results.append({
                    'frame': int(frame_idx),
                    'energy': best,
                    'receptor': receptor
                })
                print(f'  Frame {frame_idx}: {best:.2f} kcal/mol')
            else:
                print(f'  Frame {frame_idx}: no poses generated')
                
        except Exception as e:
            print(f'  Frame {frame_idx}: failed — {str(e)[:60]}')
    
    if results:
        all_docking_results[ligand_name] = results

# Save results
with open('ensemble_docking_results.json', 'w') as f:
    json.dump(all_docking_results, f, indent=2)

print(f'\n✅ Ensemble docking complete: {len(all_docking_results)} ligands with poses')

In [ ]:
#38
# Visualize docking results
import matplotlib.pyplot as plt
import pandas as pd

if all_docking_results:
    df_dock = pd.DataFrame([
        {'Compound': name, 'Best Score (kcal/mol)': data['best_score'],
         'Mean Score (kcal/mol)': data['mean_score']}
        for name, data in all_docking_results.items()
    ]).sort_values('Best Score (kcal/mol)')

    fig, ax = plt.subplots(figsize=(10, 5))
    x = range(len(df_dock))
    bars = ax.bar(x, df_dock['Best Score (kcal/mol)'], color='steelblue', label='Best pose')
    ax.scatter(x, df_dock['Mean Score (kcal/mol)'], color='red', zorder=5, label='Mean (ensemble)')
    ax.set_xticks(list(x))
    ax.set_xticklabels(df_dock['Compound'], rotation=30, ha='right')
    ax.set_ylabel('Docking score (kcal/mol)')
    ax.set_title('GPR151 Orthosteric Site — GUM-Class Ensemble Docking Results')
    ax.axhline(y=-6.0, color='orange', ls='--', lw=1, label='Typical hit threshold (-6 kcal/mol)')
    ax.legend()
    plt.tight_layout()
    plt.savefig('gpr151_pipeline/docking_scores.png', dpi=150)
    plt.show()

    print('\nDocking Results Summary:')
    print(df_dock.to_string(index=False))

## SECTION 11: Key Residue Contact Analysis
Identify the GPR151 residues most frequently contacted by GUM-class compounds across the docking ensemble. These are the pharmacophoric hotspots for lead optimization and for the patent claim on key binding residues.

In [ ]:
#40
import os
import numpy as np
import prolif as plf
import MDAnalysis as mda
from rdkit import Chem
import pandas as pd
import matplotlib.pyplot as plt
import glob

# Analyze protein-ligand interactions using ProLIF
# For each ligand's best docking pose, compute interaction fingerprint

print('Computing protein-ligand interaction fingerprints (ProLIF)...')

all_interactions = []

for lig_name, lig_path in ligand_pdbqts.items():
    # Find best docking pose file
    pose_files = glob.glob(f'gpr151_pipeline/docking_results/{lig_name.replace(" ","_")}_*.pdbqt')
    if not pose_files:
        continue

    # Use the receptor frame that gave the best score
    if lig_name in all_docking_results:
        best_score = all_docking_results[lig_name]['best_score']
        best_frame = selected_frames[np.argmin([
            all_docking_results[lig_name]['all_scores'][i]
            for i, _ in enumerate(sorted(selected_frames)[:10])
        ])]
    else:
        best_frame = sorted(selected_frames)[0]

    rec_pdb = f'gpr151_pipeline/docking_frames/frame_{best_frame:04d}.pdb'
    best_pose_file = f'gpr151_pipeline/docking_results/{lig_name.replace(" ","_")}_frame{best_frame:04d}.pdbqt'

    if not os.path.exists(best_pose_file) or not os.path.exists(rec_pdb):
        continue

    try:
        # Load receptor with MDAnalysis → ProLIF
        u_rec = mda.Universe(rec_pdb)
        rec_plf = plf.Molecule.from_mda(u_rec.select_atoms('protein'))

        # Load ligand from PDBQT via RDKit (MDA's convert_to('RDKIT') unreliable for PDBQT)
        # Convert PDBQT → PDB block first via obabel, then parse with RDKit
        import subprocess as _sp
        _pdb_block = _sp.run(
            ['obabel', best_pose_file, '-opdb', '-h'],
            capture_output=True, text=True
        ).stdout
        lig_mol = Chem.MolFromPDBBlock(_pdb_block, removeHs=False, sanitize=False)
        if lig_mol is None:
            print(f'  {lig_name}: could not parse ligand PDB block')
            continue
        try:
            Chem.SanitizeMol(lig_mol)
        except Exception:
            pass
        lig_plf = plf.Molecule(lig_mol)

        fp = plf.Fingerprint()
        fp.run_from_iterable([lig_plf], rec_plf)
        df_fp = fp.to_dataframe()

        # Extract residues with interactions
        for col in df_fp.columns:
            if df_fp[col].any():
                res_name = str(col[0])
                int_type = col[1]
                all_interactions.append({
                    'Ligand': lig_name,
                    'Residue': res_name,
                    'Interaction': int_type
                })
    except Exception as e:
        print(f'  {lig_name}: ProLIF failed — {e}')

if all_interactions:
    df_int = pd.DataFrame(all_interactions)
    print(f'\nInteractions found: {len(df_int)}')

    # Frequency of residue contacts across all ligands
    residue_freq = df_int.groupby('Residue').size().sort_values(ascending=False)
    print('\nTop contacted residues (across all GUM-class compounds):')
    for res, count in residue_freq.head(15).items():
        print(f'  {res}: {count} interactions')

    # Heatmap: residues × ligands × interaction types
    pivot = df_int.pivot_table(index='Residue', columns='Ligand', aggfunc='count', fill_value=0)
    top_residues = residue_freq.head(20).index
    pivot_top = pivot.loc[pivot.index.isin(top_residues)].fillna(0)

    fig, ax = plt.subplots(figsize=(12, 8))
    import seaborn as sns
    sns.heatmap(pivot_top.astype(float), ax=ax, cmap='YlOrRd',
                annot=True, fmt='.0f', linewidths=0.5)
    ax.set_title('GPR151 Orthosteric Residue Contacts — GUM-Class Compounds')
    ax.set_xlabel('Compound')
    ax.set_ylabel('Residue')
    plt.tight_layout()
    plt.savefig('gpr151_pipeline/contact_heatmap.png', dpi=150)
    plt.show()

    df_int.to_csv('gpr151_pipeline/all_interactions.csv', index=False)
    residue_freq.to_csv('gpr151_pipeline/key_residues_ranked.csv', header=['frequency'])
    print('\n✅ Key residue analysis complete')
else:
    print('No interactions computed — check docking output files')

## SECTION 12: N-Terminus Displacement — Steered MD Setup
Set up a steered MD experiment to model how GUM-class compounds displace the autoinhibitory N-terminus. We apply a pulling force along the collective variable defined as the distance between the best-docked GUM pose and the N-terminal domain.

In [ ]:
#42
from openmm import CustomCVForce, RMSDForce
from openmm.app import PDBFile, Simulation, AmberPrmtopFile
from openmm.unit import nanometer, picosecond, kelvin
import numpy as np

# For the steered MD, we need the full-length GPR151 (with N-terminus)
# loaded with the best-docked GUM compound placed in the orthosteric site
# This section sets up the system and defines the pulling coordinate

print('Setting up Steered MD for N-terminal displacement analysis...')
print()
print('Strategy:')
print('  1. Start from full-length GPR151 (with N-terminus intact)')
print('  2. Place best GUM compound in orthosteric pocket (from docking)')
print('  3. Apply harmonic biasing force pulling GUM into pocket')
print('  4. Monitor N-terminus RMSD as GUM penetrates deeper')
print('  5. Free energy estimate via Jarzynski equality')
print()

# CV1: GUM compound center of mass depth into pocket (Z-coordinate relative to pocket center)
# CV2: N-terminal domain RMSD from autoinhibited reference conformation
# Pulling CV1 → GUM deeper into pocket → expect CV2 (N-term RMSD) to increase

# Define the steered MD parameters
SMD_CONFIG = {
    'pulling_rate': 0.001,      # nm/ps = 1 m/s (fast for demonstration)
    'spring_constant': 1000.0,  # kJ/mol/nm²
    'total_time_ns': 5.0,       # total steered MD time
    'save_every_ps': 1.0,       # save trajectory every 1 ps
    'n_replicates': 3,          # for Jarzynski averaging
}

print('Steered MD configuration:')
for k, v in SMD_CONFIG.items():
    print(f'  {k}: {v}')

print()
print('Note: This section requires the full-length GPR151 system.')
print('The steered MD will pull GUM3/GUM4 into the pocket while')
print('tracking N-terminal domain displacement.')
print()
print('For full metadynamics free energy landscape, use PLUMED')
print('(integrated with GROMACS — see companion PLUMED input below)')

# Write PLUMED input for metadynamics (for use with GROMACS)
# protein variable may not exist if this cell runs standalone
_n_protein = len(protein) if 'protein' in dir() and protein is not None else 369  # GPR151-ΔN approx

plumed_input = '''
# PLUMED input for GPR151 N-terminal displacement metadynamics
# Collective variables:
#   cv1: GUM ligand center of mass Z-coordinate (pocket penetration depth)
#   cv2: N-terminal domain RMSD from autoinhibited reference

WHOLEMOLECULES ENTITY0=1-{n_protein} ENTITY1={lig_start}-{lig_end}

# CV1: Ligand Z-position relative to orthosteric pocket center
lig_com: COM ATOMS={lig_start}-{lig_end}
pocket_ref: FIXEDATOM AT={pocket_x},{pocket_y},{pocket_z}
cv1: DISTANCE ATOMS=lig_com,pocket_ref COMPONENTS

# CV2: N-terminal RMSD (residues 1-{tm1_minus1} Cα atoms)
nterm_ca: GROUP ATOMS=@CA:1-{tm1_minus1}
cv2: RMSD REFERENCE=gpr151_nterm_ref.pdb TYPE=OPTIMAL ATOMS=nterm_ca

# Well-tempered metadynamics
METAD ...
  ARG=cv1.z,cv2
  SIGMA=0.05,0.1
  HEIGHT=1.2
  BIASFACTOR=10
  TEMP=310
  PACE=500
  LABEL=metad
  FILE=HILLS
  GRID_MIN=-2.0,0.0
  GRID_MAX=2.0,3.0
  GRID_BIN=200,300
... METAD

PRINT ARG=cv1.z,cv2,metad.bias STRIDE=500 FILE=COLVAR
'''.format(
    n_protein=_n_protein,
    lig_start=_n_protein+1,
    lig_end=_n_protein+50,  # approximate
    pocket_x=f'{BOX_CENTER[0]/10:.3f}' if 'BOX_CENTER' in dir() else '0.000',  # convert Å to nm
    pocket_y=f'{BOX_CENTER[1]/10:.3f}' if 'BOX_CENTER' in dir() else '0.000',
    pocket_z=f'{BOX_CENTER[2]/10:.3f}' if 'BOX_CENTER' in dir() else '0.000',
    tm1_minus1=TM1_START-1
)

with open('gpr151_pipeline/plumed_metadynamics.dat', 'w') as f:
    f.write(plumed_input)
print('✅ PLUMED metadynamics input saved: gpr151_pipeline/plumed_metadynamics.dat')

## SECTION 13: CNS Drug-Like Property Screening
Filter docking hits for CNS drug-like properties required for habenula/spinal cord targeting.

In [ ]:
#44
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors
import pandas as pd

def cns_mpo_score(mol):
    """
    CNS Multiparameter Optimization score (Wager et al. 2010).
    Scores 6 properties, each contributing 0–1, max total = 6.0
    Compounds with CNS MPO > 4.0 have high probability of CNS penetrance.
    """
    mw = Descriptors.ExactMolWt(mol)
    clogp = Descriptors.MolLogP(mol)
    tpsa = Descriptors.TPSA(mol)
    hbd = Descriptors.NumHDonors(mol)
    pka_basic = 8.0  # estimated (would use pKa predictor in production)
    plogd = clogp - 0.025 * (pka_basic - 7.4) if pka_basic > 7.4 else clogp

    # Desirability functions for each property
    def desirability(value, low_good, high_good, low_ok, high_ok):
        if low_good <= value <= high_good:
            return 1.0
        elif low_ok <= value < low_good:
            return (value - low_ok) / (low_good - low_ok)
        elif high_good < value <= high_ok:
            return (high_ok - value) / (high_ok - high_good)
        return 0.0

    scores = {
        'MW': desirability(mw, 200, 360, 100, 500),
        'cLogP': desirability(clogp, 1, 3, -1, 5),
        'TPSA': desirability(tpsa, 20, 60, 0, 120),
        'HBD': desirability(hbd, 0, 1, 0, 3),
        'pKa_basic': desirability(pka_basic, 7.5, 9.5, 6, 11),
        'plogD': desirability(plogd, 1, 3, -1, 5),
    }
    total = sum(scores.values())
    return total, scores

print('CNS Drug-Like Property Analysis — GUM-Class Compounds')
print('='*65)

results = []
for name, (smiles, mol) in valid_compounds.items():
    mpo, scores = cns_mpo_score(mol)
    dock_score = all_docking_results.get(name, {}).get('best_score', 'N/A')

    results.append({
        'Compound': name,
        'MW': f"{Descriptors.ExactMolWt(mol):.1f}",
        'cLogP': f"{Descriptors.MolLogP(mol):.2f}",
        'TPSA': f"{Descriptors.TPSA(mol):.1f}",
        'HBD': Descriptors.NumHDonors(mol),
        'CNS_MPO': f"{mpo:.2f}/6.0",
        'Docking': f"{dock_score:.2f}" if isinstance(dock_score, float) else dock_score,
        'CNS_Ready': '✅' if mpo >= 4.0 else '⚠️' if mpo >= 3.0 else '❌'
    })

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

df_results.to_csv('gpr151_pipeline/compound_ranking.csv', index=False)
print('\n✅ Compound ranking saved: gpr151_pipeline/compound_ranking.csv')
print('\nThreshold: CNS MPO ≥ 4.0 for CNS drug-like profile (required for habenula targeting)')

## SECTION 14: Results Summary & Visualization
Compile all pipeline outputs into a final results summary.

In [ ]:
#46
import os
import py3Dmol
from IPython.display import display, HTML

# 3D visualization of the best-docked GUM compound in GPR151 orthosteric site
# Load the receptor and best pose

if all_docking_results:
    best_compound = min(all_docking_results, key=lambda x: all_docking_results[x]['best_score'])
    best_score_val = all_docking_results[best_compound]['best_score']
    best_frame_for_viz = sorted(selected_frames)[0]

    rec_pdb_path = f'gpr151_pipeline/docking_frames/frame_{best_frame_for_viz:04d}.pdb'
    lig_pdbqt_path = f'gpr151_pipeline/docking_results/{best_compound.replace(" ","_")}_frame{best_frame_for_viz:04d}.pdbqt'

    if os.path.exists(rec_pdb_path):
        with open(rec_pdb_path) as f:
            rec_pdb_str = f.read()

        view = py3Dmol.view(width=800, height=500)
        view.addModel(rec_pdb_str, 'pdb')
        view.setStyle({'chain': 'A'}, {
            'cartoon': {'color': 'spectrum', 'opacity': 0.8}
        })

        # Highlight orthosteric pocket region
        view.addSurface(py3Dmol.SES, {
            'opacity': 0.2,
            'color': 'lightblue'
        }, {'chain': 'A'})

        if os.path.exists(lig_pdbqt_path):
            with open(lig_pdbqt_path) as f:
                lig_str = f.read()
            view.addModel(lig_str, 'pdbqt')
            view.setStyle({'model': 1}, {'stick': {'colorscheme': 'orangeCarbon', 'radius': 0.15}})

        view.zoomTo({'chain': 'A'})
        view.show()

        print(f'\n🏆 Best compound: {best_compound}')
        print(f'   Docking score: {best_score_val:.2f} kcal/mol')
else:
    print('No docking results to visualize')

In [ ]:
#47
import os  # ensure available
# Final summary
print('='*70)
print('GPR151 COMPUTATIONAL PIPELINE — RESULTS SUMMARY')
print('='*70)
print()
print(f'Target: GPR151 (UniProt Q8TDV0) — Orphan GPCR')
print(f'Structure: AlphaFold v4 (wild-type, no thermostabilizing mutations)')
print(f'Truncation: Residues 1–{TM1_START-1} removed (autoinhibitory N-terminus)')
print(f'Membrane: POPC:POPE:Cholesterol (40:40:20), 0.15M NaCl, 310 K')
print(f'MD: {PROD_NS} ns production run, ff19SB + Lipid21 force field, OpenMM/CUDA')
print(f'Pocket snapshots for docking: {len(selected_frames)} conformations')
print()

if all_docking_results:
    print('DOCKING RESULTS (GUM-class psoralen compounds):')
    for name, data in sorted(all_docking_results.items(),
                             key=lambda x: x[1]['best_score']):
        print(f'  {name:30s}: best={data["best_score"]:.2f}, mean={data["mean_score"]:.2f} kcal/mol')

print()
print('OUTPUT FILES:')
for f in sorted(os.listdir('gpr151_pipeline')):
    path = f'gpr151_pipeline/{f}'
    if os.path.isfile(path):
        size_kb = os.path.getsize(path) / 1024
        print(f'  {f}: {size_kb:.1f} KB')

print()
print('NEXT STEPS:')
print('  1. Add exact GUM3/GUM4 SMILES from Yoshida et al. 2026 to compound library')
print('  2. Run full 500 ns MD (extend on HPC cluster — download checkpoint.chk)')
print('  3. Run metadynamics with PLUMED for N-terminus displacement free energy')
print('  4. Expand virtual screen to ZINC psoralen subset')
print('  5. Submit top candidates for in vitro GTPγS binding assay')
print('='*70)

In [ ]:
#48
# Package results — Kaggle exposes everything under /kaggle/working in the notebook's
# Output tab automatically once you commit (Save Version -> Save & Run All). No explicit
# download call needed; zip it up here for convenience anyway.
import zipfile, os

zip_path = 'GPR151_pipeline_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk('gpr151_pipeline'):
        for file in files:
            filepath = os.path.join(root, file)
            zf.write(filepath)

print(f'✅ Results packaged: {zip_path} ({os.path.getsize(zip_path)/1024/1024:.1f} MB)')
print('Available from this notebook\'s Output tab after commit, or publish gpr151_pipeline/')
print('as a Kaggle Dataset to keep it independent of this notebook version.')


In [ ]:
#49
import shutil, os

# Copy checkpoint into pipeline dir so it gets included in the dataset export
if os.path.exists('/kaggle/working/gpr151_md.checkpoint'):
    shutil.copy2('/kaggle/working/gpr151_md.checkpoint', '/kaggle/working/gpr151_pipeline/')
    print('✅ Copied checkpoint to pipeline folder')

# List what will be exported
total_size = 0
for root, dirs, files in os.walk('/kaggle/working/gpr151_pipeline'):
    for f in files:
        fp = os.path.join(root, f)
        size = os.path.getsize(fp)
        total_size += size
        if size > 1024*1024:
            print(f'  {fp}: {size/1024/1024:.1f} MB')

print(f'\nTotal: {total_size/1024/1024:.1f} MB')